In [ ]:
I have kept the existing Care Product source extraction logic unchanged.

The only change added is a mapping step after combined_care_product, where the source system instance code like MPB001, CF001, or WIP001 is mapped to its numeric ID using silver_rdm_source_system_instance.

The final join with silver_rdm_care_product now uses this mapped numeric ID with cprod_src_id, instead of using cprod_src_sys_inst_src_id.

In [ ]:
Updated the Care Product RDM add-code logic to handle source system instance mapping correctly.

Existing source extraction logic has been kept unchanged. Added a mapping step after combined_care_product to map source system instance codes such as MPB001, CF001, WIP001, etc. to their numeric IDs using silver_rdm_source_system_instance.

Updated the final comparison with silver_rdm_care_product to use the mapped numeric source system instance ID along with cprod_src_id, instead of using cprod_src_sys_inst_src_id.

This should prevent repeated duplicate values being added to silver_rdm_care_product_add due to source instance ID mismatch.

In [ ]:

mapped_care_product AS (
    SELECT
        c.cprod_name,
        c.cprod_src_sys_inst_id,
        ssi.src_sys_inst_id AS cprod_src_sys_inst_lookup_id,
        c.cprod_src_id
    FROM combined_care_product c
    LEFT JOIN silver_rdm_source_system_instance ssi
        ON UPPER(TRIM(c.cprod_src_sys_inst_id)) = UPPER(TRIM(ssi.src_sys_inst_src_id))
)

-- Keep only new records not already present in silver_rdm_care_product
SELECT
    m.cprod_name,
    m.cprod_src_sys_inst_id,
    m.cprod_src_sys_inst_lookup_id,
    m.cprod_src_id
FROM mapped_care_product m
LEFT JOIN silver_rdm_care_product r
    ON m.cprod_src_sys_inst_lookup_id = r.cprod_src_sys_inst_id
   AND UPPER(TRIM(m.cprod_src_id)) = UPPER(TRIM(r.cprod_src_id))
WHERE r.cprod_src_id IS NULL
  AND m.cprod_src_sys_inst_lookup_id IS NOT NULL

In [ ]:
SELECT
    m.cprod_name,
    m.cprod_src_sys_inst_id,
    m.cprod_src_sys_inst_lookup_id,
    m.cprod_src_id
FROM mapped_care_product m
LEFT JOIN silver_rdm_care_product r
    ON m.cprod_src_sys_inst_lookup_id = r.cprod_src_sys_inst_id
   AND UPPER(TRIM(m.cprod_src_id)) = UPPER(TRIM(r.cprod_src_id))
WHERE r.cprod_src_id IS NULL
  AND m.cprod_src_sys_inst_lookup_id IS NOT NULL

In [ ]:
WITH duplicate_check AS (
    SELECT
        *,
        COUNT(*) OVER (
            PARTITION BY
                cprod_src_name,
                cprod_src_sys_inst_id,
                cprod_src_sys_inst_src_id,
                cprod_src_id
        ) AS duplicate_count
    FROM silver_rdm_care_product
)
SELECT
    cprod_id,
    cprod_src_name,
    cprod_src_sys_inst_id,
    cprod_src_sys_inst_src_id,
    cprod_src_id,
    duplicate_count
FROM duplicate_check
WHERE duplicate_count > 1
ORDER BY
    cprod_src_name,
    cprod_src_sys_inst_id,
    cprod_src_sys_inst_src_id,
    cprod_src_id,
    cprod_id;

In [ ]:
SELECT
    cprod_src_name,
    cprod_src_sys_inst_id,
    cprod_src_sys_inst_src_id,
    cprod_src_id,
    COUNT(*) AS cnt
FROM silver_rdm_care_product
GROUP BY
    cprod_src_name,
    cprod_src_sys_inst_id,
    cprod_src_sys_inst_src_id,
    cprod_src_id
HAVING COUNT(*) > 1
ORDER BY cnt DESC;

In [ ]:
SELECT
    cprod_src_name,
    cprod_src_sys_inst_id,
    cprod_src_sys_inst_src_id,
    cprod_src_id,
    COUNT(*) AS cnt,
    COUNT(DISTINCT cprod_type_conformed) AS type_count,
    COUNT(DISTINCT cprod_group_conformed) AS group_count,
    COUNT(DISTINCT cprod_service_id) AS service_count,
    COUNT(DISTINCT crod_service_id) AS crod_service_count,
    COUNT(DISTINCT z_src_is_active) AS active_count
FROM silver_rdm_care_product
GROUP BY
    cprod_src_name,
    cprod_src_sys_inst_id,
    cprod_src_sys_inst_src_id,
    cprod_src_id
HAVING COUNT(*) > 1
ORDER BY cnt DESC;

In [ ]:
SELECT *
FROM silver_rdm_care_product
WHERE cprod_src_name = 'Acceptance and Commitment Therapy (ACT)_Face to Face'
  AND cprod_src_sys_inst_id = 1
  AND cprod_src_sys_inst_src_id = 'MPB001'
  AND cprod_src_id = 'MPB001_4_4'
ORDER BY cprod_id;

In [ ]:
when we initially started the cprod work, the source system instance ID was coming in text/source code format, so the join was kept based on that.

But during Sprint 7, after Mali made the SharePoint changes, the structure changed and we may have missed updating this join accordingly.

Now in the ADD logic, the value is still coming in text format like MPB001, IAPT277, TM3001, but in silver_rdm_care_product, cprod_src_sys_inst_id is now numeric and the text/source code is stored in cprod_src_sys_inst_src_id.

So the join is not matching because it is comparing text with numeric values. Because of that, existing care products are being treated as new records again, which is likely causing the duplicates.